# Dataset Comparison: Benchmark vs Comp Period Files

This notebook compares the `benchmark_weights_carbon_intensity_{period}.xlsx` files with their corresponding `dataset_comp_{period}.xlsx` files across all time periods.

In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Define Periods and Load Data

In [96]:
# Define periods
periods = ['1221', '0322', '0622', '0922', '1222']

# Data directory
data_dir = Path('data/datasets')

# Dictionary to store loaded data
benchmark_data = {}
comp_data = {}

# Load all datasets
for period in periods:
    # Load benchmark file
    benchmark_file = data_dir / f'benchmark_weights_carbon_intensity_{period}.xlsx'
    if benchmark_file.exists():
        benchmark_data[period] = pd.read_excel(benchmark_file)
        print(f"Loaded benchmark data for {period}: {len(benchmark_data[period])} rows")
    else:
        print(f"Warning: Benchmark file not found for {period}")
    
    # Load comp file
    comp_file = data_dir / f'dataset_comp_{period}.xlsx'
    if comp_file.exists():
        comp_data[period] = pd.read_excel(comp_file)
        print(f"Loaded comp data for {period}: {len(comp_data[period])} rows")
    else:
        print(f"Warning: Comp file not found for {period}")
    print()

Loaded benchmark data for 1221: 497 rows
Loaded comp data for 1221: 497 rows

Loaded benchmark data for 0322: 498 rows
Loaded comp data for 0322: 498 rows

Loaded benchmark data for 0622: 498 rows
Loaded comp data for 0622: 498 rows

Loaded benchmark data for 0922: 498 rows
Loaded comp data for 0922: 498 rows

Loaded benchmark data for 1222: 498 rows
Loaded comp data for 1222: 498 rows



## 2. Compare Number of Companies Per Period

In [97]:
# Create comparison dataframe
comparison_summary = pd.DataFrame({
    'Period': periods,
    'Benchmark Companies': [len(benchmark_data.get(p, [])) for p in periods],
    'Comp Dataset Companies': [len(comp_data.get(p, [])) for p in periods]
})

comparison_summary['Match'] = comparison_summary['Benchmark Companies'] == comparison_summary['Comp Dataset Companies']

print("\nCompany Count Comparison:")
print(comparison_summary)
print(f"\nAll periods match: {comparison_summary['Match'].all()}")


Company Count Comparison:
  Period  Benchmark Companies  Comp Dataset Companies  Match
0   1221                  497                     497   True
1   0322                  498                     498   True
2   0622                  498                     498   True
3   0922                  498                     498   True
4   1222                  498                     498   True

All periods match: True


## 3. Compare Common Columns Between Files

In [98]:
# Common columns between benchmark and comp datasets
common_cols = ['SYMBOL', 'NAME', 'GICS Sector', 'Carbon Intensity', 'weight_in_sector']

print("Columns in benchmark files:", benchmark_data[periods[0]].columns.tolist())
print("\nColumns in comp files:", comp_data[periods[0]].columns.tolist())
print("\nCommon columns to compare:", common_cols)

Columns in benchmark files: ['SYMBOL', 'NAME', 'GICS Sector', 'Carbon Intensity', 'weight_in_sector']

Columns in comp files: ['NAME', 'SYMBOL', 'TYPE', 'GICS Sector', 'Price last day Dec 21', 'ffnosh last day Dec 21', 'float_mcap', 'Scope 1', 'Scope 2', 'Scope 3', 'Revenue', 'Scope 1+2+3', 'Carbon Intensity', 'Scope 1 Imputed', 'Scope 2 Imputed', 'Scope 3 Imputed', 'weight_in_sector', 'rank_in_sector']

Common columns to compare: ['SYMBOL', 'NAME', 'GICS Sector', 'Carbon Intensity', 'weight_in_sector']


## 4. Detailed Comparison for Each Period

In [99]:
def compare_datasets(period, benchmark_df, comp_df):
    """
    Compare benchmark and comp datasets for a specific period
    """
    print(f"\n{'='*80}")
    print(f"Period: {period}")
    print(f"{'='*80}")
    
    # 1. Check if same companies (by SYMBOL)
    benchmark_symbols = set(benchmark_df['SYMBOL'])
    comp_symbols = set(comp_df['SYMBOL'])
    
    only_in_benchmark = benchmark_symbols - comp_symbols
    only_in_comp = comp_symbols - benchmark_symbols
    common_symbols = benchmark_symbols & comp_symbols
    
    print(f"\n1. SYMBOL Comparison:")
    print(f"   - Common symbols: {len(common_symbols)}")
    print(f"   - Only in benchmark: {len(only_in_benchmark)}")
    if only_in_benchmark:
        print(f"     {only_in_benchmark}")
    print(f"   - Only in comp: {len(only_in_comp)}")
    if only_in_comp:
        print(f"     {only_in_comp}")
    
    # 2. Compare common columns for matching symbols
    if len(common_symbols) > 0:
        # Merge on SYMBOL
        merged = pd.merge(
            benchmark_df[common_cols],
            comp_df[common_cols],
            on='SYMBOL',
            suffixes=('_benchmark', '_comp')
        )
        
        print(f"\n2. Column-by-Column Comparison:")
        
        # Compare GICS Sector
        sector_mismatch = merged[merged['GICS Sector_benchmark'] != merged['GICS Sector_comp']]
        print(f"   - GICS Sector mismatches: {len(sector_mismatch)}")
        if len(sector_mismatch) > 0:
            print(sector_mismatch[['SYMBOL', 'GICS Sector_benchmark', 'GICS Sector_comp']])
        
        # Compare Carbon Intensity (numerical - allow small differences)
        ci_diff = abs(merged['Carbon Intensity_benchmark'] - merged['Carbon Intensity_comp'])
        ci_mismatch = merged[ci_diff > 0.0001]
        print(f"\n   - Carbon Intensity mismatches (diff > 0.0001): {len(ci_mismatch)}")
        if len(ci_mismatch) > 0:
            print(ci_mismatch[['SYMBOL', 'Carbon Intensity_benchmark', 'Carbon Intensity_comp']])
            print(f"     Max difference: {ci_diff.max():.6f}")
            print(f"     Mean difference: {ci_diff.mean():.6f}")
        
        # Compare weights
        weight_diff = abs(merged['weight_in_sector_benchmark'] - merged['weight_in_sector_comp'])
        weight_mismatch = merged[weight_diff > 0.0001]
        print(f"\n   - Weight in sector mismatches (diff > 0.0001): {len(weight_mismatch)}")
        if len(weight_mismatch) > 0:
            print(weight_mismatch[['SYMBOL', 'weight_in_sector_benchmark', 'weight_in_sector_comp']])
            print(f"     Max difference: {weight_diff.max():.6f}")
            print(f"     Mean difference: {weight_diff.mean():.6f}")
        
        # 3. Summary statistics comparison
        print(f"\n3. Summary Statistics:")
        stats_comparison = pd.DataFrame({
            'Metric': ['Mean Carbon Intensity', 'Std Carbon Intensity', 'Mean Weight', 'Sum of Weights'],
            'Benchmark': [
                benchmark_df['Carbon Intensity'].mean(),
                benchmark_df['Carbon Intensity'].std(),
                benchmark_df['weight_in_sector'].mean(),
                benchmark_df['weight_in_sector'].sum()
            ],
            'Comp': [
                comp_df['Carbon Intensity'].mean(),
                comp_df['Carbon Intensity'].std(),
                comp_df['weight_in_sector'].mean(),
                comp_df['weight_in_sector'].sum()
            ]
        })
        stats_comparison['Difference'] = stats_comparison['Benchmark'] - stats_comparison['Comp']
        print(stats_comparison)
        
        return merged
    
    return None

# Run comparison for all periods
merged_data = {}
for period in periods:
    if period in benchmark_data and period in comp_data:
        merged_data[period] = compare_datasets(period, benchmark_data[period], comp_data[period])


Period: 1221

1. SYMBOL Comparison:
   - Common symbols: 497
   - Only in benchmark: 0
   - Only in comp: 0

2. Column-by-Column Comparison:
   - GICS Sector mismatches: 0

   - Carbon Intensity mismatches (diff > 0.0001): 0

   - Weight in sector mismatches (diff > 0.0001): 0

3. Summary Statistics:
                  Metric  Benchmark       Comp    Difference
0  Mean Carbon Intensity   1.748549   1.748549  2.220446e-16
1   Std Carbon Intensity   5.372123   5.372123 -2.664535e-15
2            Mean Weight   0.022133   0.022133  0.000000e+00
3         Sum of Weights  11.000000  11.000000  0.000000e+00

Period: 0322

1. SYMBOL Comparison:
   - Common symbols: 498
   - Only in benchmark: 0
   - Only in comp: 0

2. Column-by-Column Comparison:
   - GICS Sector mismatches: 0

   - Carbon Intensity mismatches (diff > 0.0001): 0

   - Weight in sector mismatches (diff > 0.0001): 0

3. Summary Statistics:
                  Metric  Benchmark       Comp  Difference
0  Mean Carbon Intensity   1.5

In [100]:
import pandas as pd

# Load the benchmark and comp datasets for period 0622
df_bench_0622 = pd.read_excel("data/datasets/benchmark_weights_carbon_intensity_0622.xlsx")
df_comp_0622 = pd.read_excel("data/datasets/dataset_comp_0622.xlsx")

print(f"Loaded df_bench_0622: {df_bench_0622.shape}")
print(f"Loaded df_comp_0622: {df_comp_0622.shape}")

# Check if OGN is in the datasets
print(f"\nOGN in benchmark: {('OGN' in df_bench_0622['SYMBOL'].values)}")
print(f"OGN in comp: {('OGN' in df_comp_0622['SYMBOL'].values)}")

# Make clean copies without OGN
bench_no_ogn = df_bench_0622[df_bench_0622['SYMBOL'] != 'OGN'].copy()
comp_no_ogn  = df_comp_0622[df_comp_0622['SYMBOL'] != 'OGN'].copy()

print(f"\nAfter removing OGN:")
print(f"bench_no_ogn: {bench_no_ogn.shape}")
print(f"comp_no_ogn: {comp_no_ogn.shape}")

# Align on SYMBOL, just in case
bench_no_ogn = bench_no_ogn.set_index('SYMBOL').sort_index()
comp_no_ogn  = comp_no_ogn.set_index('SYMBOL').sort_index()
common = bench_no_ogn.index.intersection(comp_no_ogn.index)

print(f"\nCommon symbols: {len(common)}")

# Compute differences again
carbon_diff = (bench_no_ogn.loc[common, 'Carbon Intensity']
               - comp_no_ogn.loc[common, 'Carbon Intensity']).abs()

print(f"\nCarbon Intensity Comparison:")
print(f"Count with diff > 0.0001: {(carbon_diff > 0.0001).sum()}")
print(f"Mean diff: {carbon_diff.mean():.6f}")
print(f"Max diff: {carbon_diff.max():.6f}")

# Summary stats
print("\nSummary (OGN removed)")
print("Benchmark mean:", bench_no_ogn.loc[common, 'Carbon Intensity'].mean())
print("Comp mean:",      comp_no_ogn.loc[common, 'Carbon Intensity'].mean())
print("Difference:",     bench_no_ogn.loc[common, 'Carbon Intensity'].mean()
                        - comp_no_ogn.loc[common, 'Carbon Intensity'].mean())

# Show any rows with significant differences
if (carbon_diff > 0.0001).any():
    print("\n⚠ Symbols with significant Carbon Intensity differences:")
    sig_diff = carbon_diff[carbon_diff > 0.0001].sort_values(ascending=False)
    for symbol, diff in sig_diff.items():
        bench_val = bench_no_ogn.loc[symbol, 'Carbon Intensity']
        comp_val = comp_no_ogn.loc[symbol, 'Carbon Intensity']
        print(f"  {symbol}: Bench={bench_val:.6f}, Comp={comp_val:.6f}, Diff={diff:.6f}")
else:
    print("\n✓ All Carbon Intensity values match (within 0.0001 tolerance)")

Loaded df_bench_0622: (498, 5)
Loaded df_comp_0622: (498, 18)

OGN in benchmark: False
OGN in comp: False

After removing OGN:
bench_no_ogn: (498, 5)
comp_no_ogn: (498, 18)

Common symbols: 498

Carbon Intensity Comparison:
Count with diff > 0.0001: 0
Mean diff: 0.000000
Max diff: 0.000000

Summary (OGN removed)
Benchmark mean: 1.8416627268393027
Comp mean: 1.8416627268393027
Difference: 0.0

✓ All Carbon Intensity values match (within 0.0001 tolerance)


In [ ]:
import pandas as pd

# Load the benchmark and comp datasets for period 0622
df_bench_0922 = pd.read_excel("data/datasets/benchmark_weights_carbon_intensity_0922.xlsx")
df_comp_0922 = pd.read_excel("data/datasets/dataset_comp_0922.xlsx")

print(df_comp_0922.loc[df_comp_0922['SYMBOL'] == 'GOOGL', 'weight_in_sector'])
print(df_bench_0922.loc[df_bench_0922['SYMBOL'] == 'GOOGL', 'weight_in_sector'])

497    0.452789
Name: weight_in_sector, dtype: float64
0    0.452789
Name: weight_in_sector, dtype: float64


In [115]:
df_bench_0622 = pd.read_excel("data/datasets/benchmark_weights_carbon_intensity_0622.xlsx")
df_comp_0622 = pd.read_excel("data/datasets/dataset_comp_0622.xlsx")

print(df_comp_0922.loc[df_comp_0622['SYMBOL'] == 'GOOGL'])
print(df_bench_0922.loc[df_bench_0622['SYMBOL'] == 'GOOGL'])

             NAME SYMBOL    TYPE             GICS Sector  \
497  ALPHABET 'A'  GOOGL  29026M  Communication Services   

     Price last day Sep 22  ffnosh last day Sep 22    float_mcap  Scope 1  \
497              95.896387                10850560  1.040530e+09  91200.0   

       Scope 2     Scope 3    Revenue  Scope 1+2+3  Carbon Intensity  \
497  8045400.0  10034000.0  282836000   18170600.0          0.064244   

     Scope 1 Imputed  Scope 2 Imputed  Scope 3 Imputed  weight_in_sector  \
497                0                0                0          0.452789   

     rank_in_sector  
497               1  
  SYMBOL          NAME             GICS Sector  Carbon Intensity  \
0  GOOGL  ALPHABET 'A'  Communication Services          0.064244   

   weight_in_sector  
0          0.452789  


In [119]:
df_bench_0622.loc[df_bench_0622['GICS Sector'] == 'Communication Services']

,SYMBOL,NAME,GICS Sector,Carbon Intensity,weight_in_sector
0,GOOGL,ALPHABET 'A',Communication Services,0.064244,0.448552
1,META,META PLATFORMS A,Communication Services,0.106808,0.130781
2,VZ,VERIZON COMMUNICATIONS,Communication Services,0.132817,0.078126
3,CMCSA,COMCAST A,Communication Services,0.104737,0.066525
4,DIS,WALT DISNEY,Communication Services,0.143052,0.058862
5,T,AT&T,Communication Services,0.157401,0.055537
6,TMUS,T-MOBILE US,Communication Services,0.128664,0.032333
7,NFLX,NETFLIX,Communication Services,0.037565,0.026574
8,ATVI,ACTIVISION BLIZZARD DEAD - DELIST.16/10/23,Communication Services,0.036744,0.020656
9,CHTR,CHARTER COMMS.CL.A,Communication Services,0.045214,0.019312


In [120]:
df_bench_0922.loc[df_bench_0922['GICS Sector'] == 'Communication Services', ['SYMBOL', 'NAME', 'GICS Sector', 'Carbon Intensity', 'weight_in_sector']]

,SYMBOL,NAME,GICS Sector,Carbon Intensity,weight_in_sector
0,GOOGL,ALPHABET 'A',Communication Services,0.064244,0.452789
1,META,META PLATFORMS A,Communication Services,0.106808,0.128664
2,DIS,WALT DISNEY,Communication Services,0.143052,0.068753
3,VZ,VERIZON COMMUNICATIONS,Communication Services,0.132817,0.059252
4,CMCSA,COMCAST A,Communication Services,0.104737,0.058193
5,T,AT&T,Communication Services,0.157401,0.047568
6,NFLX,NETFLIX,Communication Services,0.037565,0.041821
7,TMUS,T-MOBILE US,Communication Services,0.128664,0.037626
8,ATVI,ACTIVISION BLIZZARD DEAD - DELIST.16/10/23,Communication Services,0.036744,0.023361
9,CHTR,CHARTER COMMS.CL.A,Communication Services,0.045214,0.014399


## 5. Sector-Level Comparison

In [107]:
communication_services_0622 = pd.read_excel("data/log_returns/sector_log_returns_comp_0622.xlsx", sheet_name="Communication Services")
communication_services_0622_new = pd.read_excel("data/log_returns/sector_log_returns_comp_0622_new.xlsx", sheet_name="Communication Services")

In [108]:
set(communication_services_0622_new.columns) - set(communication_services_0622.columns)

set()

In [109]:
set(communication_services_0622.columns) - set(communication_services_0622_new.columns)

set()